In [2]:
# Auto-switch to repo root so relative paths (configs/, scripts/, data/) work
from pathlib import Path
import os

_cwd = Path.cwd()
_root = _cwd
while _root != _root.parent:
    if (_root / "configs" / "config.yaml").exists():
        break
    _root = _root.parent
os.chdir(_root)
print(f"Repo root: {_root}")


Repo root: c:\Users\karki\Downloads\main_openworld\openworld


# 05 — Final open-world evaluation (Phase 5)

Runs on the **fixed** val-test split only — never tuned here.

Steps:
1. Detector-only base AP50 (experiment A) on val-test.
2. RPN + CLIP open-world evaluation: base / novel / generalized AP50 with bootstrap CIs.
3. Confusion matrix and failure gallery.


Experiment A: detector base AP50 on val-test.

In [3]:
%run scripts/train_detector.py --mode eval --config configs/config.yaml \
    --checkpoint outputs/checkpoints/epoch_3.pt \
    --image-ids data/splits/val_test_image_ids.json \
    --val-annotations data/annotations/instances_val_test2017.stripped.json \
    --images-root data/images/val2017 \
    --save-predictions outputs/predictions/detector_val_test.json


usage: train_detector.py [-h] [--mode {train,eval}] [--config CONFIG]
                         [--train-ids TRAIN_IDS] [--val-ids VAL_IDS]
                         [--train-annotations TRAIN_ANNOTATIONS]
                         [--val-annotations VAL_ANNOTATIONS]
                         [--images-root IMAGES_ROOT]
                         [--checkpoint-dir CHECKPOINT_DIR]
                         [--log-file LOG_FILE]
                         [--save-predictions SAVE_PREDICTIONS]
                         [--checkpoint CHECKPOINT] [--epochs EPOCHS]
                         [--batch-size BATCH_SIZE] [--lr LR]
                         [--limit-images LIMIT_IMAGES]
train_detector.py: error: unrecognized arguments: --image-ids data/splits/val_test_image_ids.json


SystemExit: 2

Experiment C: RPN + CLIP on val-test (the open-world pipeline). Adjust paths to the checkpoint and proposal/CLIP outputs you actually produced.

In [4]:
%run scripts/evaluate_openworld.py \
    --predictions outputs/clip/val_test_proposals/predictions.json \
    --annotations data/annotations/instances_val_test2017.stripped.json \
    --selected-categories data/splits/selected_categories.json \
    --image-ids data/splits/val_test_image_ids.json \
    --out-dir outputs/metrics


FileNotFoundError: [Errno 2] No such file or directory: 'outputs/clip/val_test_proposals/predictions.json'

In [5]:
from IPython.display import Image, display
import json

m = json.load(open("outputs/metrics/openworld_eval.json"))
print(json.dumps({k: v for k, v in m.items() if k != "per_class_AP50"}, indent=2))
print()
print("Per-class AP50:")
for name, v in m["per_class_AP50"].items():
    print(f"  {name:12s} {v:.4f}")

display(Image(filename="outputs/metrics/confusion_matrix.png"))


FileNotFoundError: [Errno 2] No such file or directory: 'outputs/metrics/openworld_eval.json'

Failure gallery — inspect why detections failed.

In [6]:
%run scripts/analyze_results.py \
    --predictions outputs/clip/val_test_proposals/predictions.json \
    --annotations data/annotations/instances_val_test2017.stripped.json \
    --images-root data/images/val2017 \
    --selected-categories data/splits/selected_categories.json \
    --image-ids data/splits/val_test_image_ids.json \
    --out-dir outputs/analysis

from IPython.display import display
from pathlib import Path

for p in sorted(Path("outputs/analysis/gallery").glob("*.png"))[:6]:
    display(Image(filename=str(p)))


FileNotFoundError: [Errno 2] No such file or directory: 'outputs/clip/val_test_proposals/predictions.json'